# The timestamp-specific positional penalty, per utterance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgrfhyL/audio_model_initial_testing/blob/main/Colab_DeltaSweep.ipynb)

Corpus WER told us *that* late-placed utterances degrade with timestamps on. It cannot tell us
**which utterances**, or how the damage is distributed. This notebook measures a paired
per-utterance quantity instead.

For each utterance `m`:

```
delta_m = [ WER(m, 25s, ts on)  - WER(m, 5s, ts on)  ]
        - [ WER(m, 25s, ts off) - WER(m, 5s, ts off) ]
```

A **difference in differences**. The inner brackets are how much moving from 5 s to 25 s costs
that utterance, with and without timestamps. Subtracting removes any position effect that exists
independently of timestamps, along with every per-utterance confound that is constant across
offsets - speaker, sentence, duration, difficulty. What survives is the part of the positional
penalty that **only exists because timestamps are enabled**, isolated per utterance.

`delta_m > 0` means timestamps made the late placement worse for that utterance.

### Why 5 s and not 0 s as the baseline

Offset 0 is the one position whose mel is *not* a pure shift of the others:
`torch.stft(center=True)` reflect-pads 200 samples, so at offset 0 that padding mirrors the
utterance's own opening instead of zeros, perturbing mel frames 0-1. Offsets 5 s and 25 s are
both bit-identical shifts of each other, so the difference is clean.

### Corpus

**All 1000 clips** of the seed-0 draw - the same corpus `Aug_23.ipynb` used, so per-utterance
results join directly to the existing offset results by `path`. 168 speakers at 5-6 utterances
each, all 8 dialect regions, 482 distinct sentences.

1000 clips x 2 offsets x 2 timestamp arms x 5 models = **20 000 decodes**, ~63 min on a T4.
Roughly 43 min of that is decoding (half of it `large-v3` alone), ~14 min is copying the corpus
off Drive - a one-time cost per session, skipped on a re-run - and the rest is installs and
checkpoint downloads.

`large-v3` at batch 16 needs ~3.1 GB of weights plus ~3.9 GB of cross-attention KV cache
(32 layers x 1500 frames x 1280 x k/v x fp16). That fits a T4's 15 GB, but with much less headroom
than `medium`, so the whole sweep runs at batch 8 rather than
leaving `large-v3` to discover the limit by OOM.
`large-v3` also uses **128** mel bins rather than 80; `n_mels` is read from `model.dims`, and the
gate in section 4 checks both filterbank sizes.

### What to expect

`delta_m` is **zero-inflated and heavy-tailed**. Computed over all 1000 clips from the local
`base` run: 782 exactly 0, 171 positive, 47 negative, mean `+0.226`, median `0.000`, max `+34.5`.
Mean and median therefore say completely different things, so this notebook reports prevalence
(how many utterances are affected) and severity (how much) separately.

## 1. Environment

Same base settings as the scaling sweep: Colab GPU, fp16 on CUDA, batch 16, `openai-whisper`.

In [ ]:
!pip -q install openai-whisper jiwer soundfile

import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Provenance

Everything needed to reproduce or audit this run, recorded into `PROVENANCE` and written next to
the results. Six things are pinned:

1. **Package versions** - `openai-whisper`, torch, numpy, soundfile, jiwer, Python, CUDA.
2. **Code version** - SHA-256 of the three `whisper` source files that define the decode path
   (`audio.py`, `decoding.py`, `model.py`), plus the current commit of this repository. Package
   version strings are coarse; hashing the actual code that runs is not.
3. **Checkpoint hashes** - `whisper` embeds each checkpoint's expected SHA-256 in its download URL
   and verifies it on fetch. Both the expected and the recomputed on-disk digest are recorded, so
   the weights are provably the official ones.
4. **Precision and device** - fp16, CUDA, GPU model, driver.
5. **Decoding options** - the exact `DecodingOptions` used, verbatim.
6. **Corpus and reference digests** - the per-clip `sha256_audio` from `corpus_digests.json`, an
   aggregate digest over the 300 selected clips, and a digest over the *normalized reference
   transcripts*. The last one pins exactly which references were scored against **without storing
   any transcript text**.

### Licensing

TIMIT is LDC93S1: licensed, not redistributable. This notebook is written so that **no transcript
text and no audio ever reaches its printed output**, because saving a Colab notebook back to
GitHub commits its outputs. Only counts, digests and numeric scores are printed. The per-utterance
results file that is safe to commit carries no text column, and is asserted to be so in section 7;
the full record, which does contain hypotheses, is written to Drive only.

In [ ]:
import hashlib, json, os, platform, sys

DRIVE_ROOT  = "/content/drive/MyDrive/NAACL"
RESULTS_CSV = os.path.join(DRIVE_ROOT, "delta_results_full.csv")     # has text -> Drive only
SAFE_CSV    = os.path.join(DRIVE_ROOT, "delta_per_utterance.csv")    # numbers only -> git-safe
PROV_JSON   = os.path.join(DRIVE_ROOT, "delta_provenance.json")
LOCAL_AUDIO = "/content/corpus"
REPO        = "AgrfhyL/audio_model_initial_testing"

def sha256_file(path, buf=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(buf), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

import numpy, soundfile, jiwer, whisper

# --- 2. code version: hash the whisper source files that define the decode path ---
_wdir = os.path.dirname(whisper.__file__)
CODE_DIGESTS = {f: sha256_file(os.path.join(_wdir, f))[:16]
                for f in ("audio.py", "decoding.py", "model.py")}

# repo commit (best effort -- Colab has no git checkout)
try:
    import urllib.request
    with urllib.request.urlopen(
            f"https://api.github.com/repos/{REPO}/commits/main", timeout=10) as r:
        REPO_COMMIT = json.load(r)["sha"]
except Exception as e:
    REPO_COMMIT = f"unavailable ({type(e).__name__})"

PROVENANCE = {
    "experiment": "per-utterance timestamp-specific positional penalty (delta_m)",
    "metric": "delta_m = [WER(25s,on)-WER(5s,on)] - [WER(25s,off)-WER(5s,off)]",
    "packages": {
        "python": platform.python_version(), "openai-whisper": whisper.__version__,
        "torch": torch.__version__, "numpy": numpy.__version__,
        "soundfile": soundfile.__version__, "jiwer": getattr(jiwer, "__version__", "n/a"),
        "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    },
    "code_version": {"whisper_source_sha256_16": CODE_DIGESTS, "repo": REPO,
                     "repo_commit": REPO_COMMIT},
    "device": {"name": torch.cuda.get_device_name(0),
               "capability": ".".join(map(str, torch.cuda.get_device_capability(0))),
               "precision": "fp16 (encoder/decoder); mel computed fp32 on CPU"},
}
print(json.dumps(PROVENANCE, indent=1))

## 3. Corpus: all 1000 clips

`corpus_digests.json` is ordered by the original seed-0 draw; `[0:1000]` is the whole of it, the
same corpus `Aug_23.ipynb` scored, so per-utterance results here join to those by `path`. Each
clip is copied to local disk (Drive's FUSE mount is far too slow to read repeatedly), decoded,
and checked against its frozen `sha256_audio` - a digest over the decoded float32 samples, so it
pins the array that actually reaches the mel.

Copying and verifying 1000 clips off Drive takes ~14 min and is the single largest fixed cost of
the run; it is skipped on a re-run in the same session, since `/content/corpus` persists.

Reference transcripts are read from the `.TXT` files in *your* TIMIT copy. They are hashed into a
reference digest and then never printed.

In [ ]:
import collections, glob, shutil, time
import numpy as np
import soundfile as sf

SLICE_LO, SLICE_HI = 0, 1000          # the full corpus

cands = [d for d in glob.glob(os.path.join(DRIVE_ROOT, "timit", "**", "TEST"), recursive=True)
         if os.path.isdir(os.path.join(d, "DR1"))]
assert cands, f"no TIMIT TEST/DR1 found under {DRIVE_ROOT}/timit"
TIMIT_TEST = sorted(cands, key=len)[0]

if not os.path.exists("corpus_digests.json"):
    !wget -q https://raw.githubusercontent.com/AgrfhyL/audio_model_initial_testing/main/corpus_digests.json
DIG = json.load(open("corpus_digests.json"))

def sha256_audio(a):
    return hashlib.sha256(np.ascontiguousarray(a, dtype=np.float32).tobytes()).hexdigest()

def load_reference(wav_path):
    with open(wav_path[:-4] + ".TXT") as f:
        return f.read().strip().split(None, 2)[2]

FILES, AUDIO, REFTEXT, bad = [], {}, {}, []
t0 = time.time()
for r in DIG["files"][SLICE_LO:SLICE_HI]:
    src = os.path.join(TIMIT_TEST, r["path"]); dst = os.path.join(LOCAL_AUDIO, r["path"])
    if not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy2(src, dst)
    a, sr = sf.read(dst, dtype="float32")
    if sr != DIG["sample_rate"] or len(a) != r["samples"] or sha256_audio(a) != r["sha256_audio"]:
        bad.append(r["path"]); continue
    AUDIO[r["path"]] = a; REFTEXT[r["path"]] = load_reference(src); FILES.append(r)

assert not bad, f"{len(bad)} clips fail their frozen digest, e.g. {bad[:3]}"
assert len(FILES) == SLICE_HI - SLICE_LO

spk = collections.Counter(r["speaker"] for r in FILES)
print(f"corpus slice [{SLICE_LO}:{SLICE_HI}] verified in {time.time()-t0:.0f}s")
print(f"  {len(FILES)} clips | {len(spk)} speakers ({min(spk.values())}-{max(spk.values())} each)"
      f" | {sum(r['sec'] for r in FILES)/60:.1f} min")
print(f"  regions {dict(sorted(collections.Counter(r['region'] for r in FILES).items()))}")

assert {r["path"] for r in FILES} == {r["path"] for r in DIG["files"][SLICE_LO:SLICE_HI]}
print(f"  the full seed-0 draw [{SLICE_LO}:{SLICE_HI}] -- same corpus as Aug_23.ipynb")

## 4. Controls, digests and the mel gate

Base settings are held identical to the scaling sweep. The corpus digest is an aggregate over the
300 selected clips' individual `sha256_audio` values; the reference digest is taken over
`path \t normalized_reference` lines, so it pins exactly what was scored **without retaining the
text**.

The mel gate is the control the whole experiment rests on: offsets 5 s and 25 s must be exact
integer-frame shifts of one another, or `delta_m` would be measuring a different spectrogram
rather than a different position. It is checked at **both** filterbank sizes, 80 and 128, since
`large-v3` uses 128 mel bins where the smaller models use 80.

In [ ]:
from whisper.audio import log_mel_spectrogram, N_SAMPLES, SAMPLE_RATE, HOP_LENGTH
from whisper.normalizers import EnglishTextNormalizer

MODELS      = ["tiny", "base", "small", "medium", "large-v3"]
OFFSETS_S   = [5, 25]                       # baseline and late placement
OFFSETS     = [s * SAMPLE_RATE for s in OFFSETS_S]
TS_ARMS     = [True, False]
# One batch size for every model. 8 divides 1000 exactly (125 x 8), so no model ends on a
# short remainder batch, and it fits large-v3 on a T4 with room to spare. Batch size is a
# nuisance parameter; holding it constant costs ~3 min and removes it as a variable.
BATCH       = 8

DECODE_OPTS = dict(task="transcribe", language="en", temperature=0.0, beam_size=None,
                   best_of=None, prompt=None, prefix=None, fp16=True,
                   suppress_blank=True, suppress_tokens="-1", max_initial_timestamp=1.0)

normalizer = EnglishTextNormalizer()
REF = {p: normalizer(t) for p, t in REFTEXT.items()}
assert all(v.strip() for v in REF.values())

# --- digests: corpus and references (no text retained) ---
corpus_digest = sha256_bytes("\n".join(
    f"{r['path']}\t{r['sha256_audio']}" for r in FILES).encode())
reference_digest = sha256_bytes("\n".join(
    f"{r['path']}\t{REF[r['path']]}" for r in FILES).encode())

PROVENANCE["decoding"] = {**DECODE_OPTS, "greedy": True,
                          "condition_on_previous_text": "N/A - decode(); prompt=None",
                          "offsets_s": OFFSETS_S, "batch": BATCH}
PROVENANCE["corpus"] = {
    "source": "TIMIT (LDC93S1) TEST -- licensed, not redistributable",
    "spec_file": "corpus_digests.json",
    "spec_sha256": sha256_file("corpus_digests.json"),
    "slice": [SLICE_LO, SLICE_HI], "n_clips": len(FILES),
    "n_speakers": len({r["speaker"] for r in FILES}),
    "corpus_digest_sha256": corpus_digest,
    "reference_digest_sha256": reference_digest,
    "reference_normalizer": "whisper EnglishTextNormalizer",
    "note": "digests pin the audio arrays and the normalized references; no text is stored",
}
print("corpus digest   ", corpus_digest)
print("reference digest", reference_digest)

# --- placement + mel gate ---
def place(audio, off_samples):
    buf = np.zeros(N_SAMPLES, dtype=np.float32)
    buf[off_samples:off_samples + len(audio)] = audio
    return buf

def mel_of(audio, off_samples, n_mels):
    return log_mel_spectrogram(torch.from_numpy(place(audio, off_samples)), n_mels)

for s, n in zip(OFFSETS_S, OFFSETS):
    assert n % HOP_LENGTH == 0
for r in FILES:
    for off in OFFSETS:
        assert off + r["samples"] <= N_SAMPLES, (r["path"], off)

import random as _random
FPS, NF = SAMPLE_RATE // HOP_LENGTH, N_SAMPLES // HOP_LENGTH

# both filterbank sizes: tiny..medium use 80 mels, large-v3 and turbo use 128
gate = {}
for n_mels in (80, 128):
    worst = 0.0
    for r in _random.Random(7).sample(FILES, 40):
        a = AUDIO[r["path"]]; nf = int(np.ceil(len(a) / HOP_LENGTH)) + 2
        base = mel_of(a, 5 * SAMPLE_RATE, n_mels).numpy()
        seg = mel_of(a, 25 * SAMPLE_RATE, n_mels).numpy()
        n = min(nf, NF - 25 * FPS)
        worst = max(worst, float(np.abs(seg[:, 25*FPS:25*FPS+n]
                                        - base[:, 5*FPS:5*FPS+n]).max()))
    gate[n_mels] = worst
    assert worst == 0.0, f"mel gate FAILED at n_mels={n_mels}: differ by {worst:.3e}"
PROVENANCE["mel_gate"] = {"offsets_compared": [5, 25], "n_sampled": 40,
                          "max_abs_dev_by_n_mels": gate}
print("mel gate: 5 s vs 25 s bit-identical on 40 sampled clips at "
      + ", ".join(f"n_mels={k} (|dev| {v:.1e})" for k, v in gate.items()))

## 5. The sweep

20 000 decodes. Model-outer so each checkpoint is loaded once; its SHA-256 is recorded against
the expected digest from the download URL at load time. A single batch size (`BATCH = 8`) is used for every model,
with an OOM fallback retained only as an emergency net. Both timestamp arms decode off one shared mel,
which guarantees they see bit-identical input.

Resumable: rows append to Drive and any `(model, path, offset, arm)` already present is skipped.

### A note on batch size

Batch size is a nuisance parameter, not part of the experiment, so it is held **constant at 8 for
every model**. 8 divides 1000 exactly (125 x 8), so no model ends on a short remainder batch
either — at batch 16 the four smaller models would each finish with a ragged batch of 8. It also
fits `large-v3` on a T4 with room to spare (~3.1 GB weights plus ~2 GB of cross-attention KV
cache), so the memory limit is never discovered by hitting it.

Two things are enforced rather than hoped for. An assertion fails the run if the corpus size is
not divisible by `BATCH`. And resume works at **(model, offset)** granularity rather than per
utterance: a partially finished offset is re-decoded in full, so every batch is always formed from
the same 1000-clip list — otherwise an interrupted run would resume with ragged batches. Rows
already on disk are not rewritten.

The OOM fallback is kept as an emergency net, but taking it is treated as a defect: section 9
reports a `WARN` rather than a `PASS` if any split occurred, and tells you to lower `BATCH` and
re-run that model.

Whether batch size can change the output at all was checked directly on `base`: decoding the same
16 utterances at batch 16, 8, 4 and 1 gives **byte-identical** transcripts in both fp32 and fp16.
That test ran on Apple MPS, not CUDA, so it is evidence rather than proof for the Colab runtime —
which is exactly why the batch is pinned instead of left to vary.

In [ ]:
import csv, gc

FULL_FIELDS = ["model", "n_mels", "path", "speaker", "offset_s", "timestamps",
               "text", "avg_logprob", "no_speech_prob"]

def checkpoint_digest(name):
    '''Expected SHA-256 (embedded in whisper's download URL) and the on-disk digest.'''
    url = whisper._MODELS[name]
    expected = url.split("/")[-2]
    root = os.path.join(os.getenv("XDG_CACHE_HOME",
                                  os.path.join(os.path.expanduser("~"), ".cache")), "whisper")
    p = os.path.join(root, os.path.basename(url))
    return expected, (sha256_file(p) if os.path.exists(p) else None)

split_events = []


def decode_resilient(model, mel, opts, tag):
    """whisper.decode, halving the batch and retrying on CUDA OOM.

    large-v3 at batch 16 needs roughly 3.1 GB of weights plus ~3.9 GB of cross-attention
    KV cache (32 layers x 1500 frames x 1280 x k/v x fp16), which fits a T4 but with far
    less headroom than medium. Splitting on demand beats guessing a smaller batch for
    every model.
    """
    try:
        return whisper.decode(model, mel, opts)
    except torch.cuda.OutOfMemoryError:
        if mel.shape[0] == 1:
            raise
        torch.cuda.empty_cache()
        half = mel.shape[0] // 2
        split_events.append((tag, int(mel.shape[0])))
        return (decode_resilient(model, mel[:half], opts, tag)
                + decode_resilient(model, mel[half:], opts, tag))


# no remainder batch: every batch is exactly BATCH wide
assert len(FILES) % BATCH == 0, (
    f"{len(FILES)} clips is not divisible by BATCH={BATCH}; the final batch would be "
    f"{len(FILES) % BATCH} wide, making batching non-uniform")

done = set()
if os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV, newline="") as f:
        for row in csv.DictReader(f):
            done.add((row["model"], row["path"], int(row["offset_s"]),
                      row["timestamps"] == "on"))
total = len(MODELS) * len(FILES) * len(OFFSETS_S) * len(TS_ARMS)
print(f"{len(done)}/{total} cells already done")

os.makedirs(DRIVE_ROOT, exist_ok=True)
ckpt, t_start = {}, time.time()
with open(RESULTS_CSV, "a", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=FULL_FIELDS)
    if not done:
        w.writeheader()

    for name in MODELS:
        model = whisper.load_model(name, device="cuda")
        exp, act = checkpoint_digest(name)
        ckpt[name] = {"expected_sha256": exp, "ondisk_sha256": act, "verified": exp == act,
                      "params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
                      "n_mels": model.dims.n_mels,
                      "n_audio_layer": model.dims.n_audio_layer,
                      "n_text_layer": model.dims.n_text_layer}
        assert ckpt[name]["verified"], f"{name}: checkpoint digest mismatch"
        n_mels = model.dims.n_mels
        ckpt[name]["batch"] = BATCH
        print(f"\n{name}: {ckpt[name]['params_M']}M params, n_mels={n_mels}, "
              f"sha256 {act[:16]}... verified, batch {BATCH}")

        for s, off in zip(OFFSETS_S, OFFSETS):
            # Resume at (model, offset) granularity, not per utterance: a partially
            # finished offset is re-decoded in full so every batch is formed from the
            # same 1000-clip list. Rows already present are still not rewritten.
            if all((name, r["path"], s, t) in done for r in FILES for t in TS_ARMS):
                continue
            pending = FILES
            t0 = time.time()
            for i in range(0, len(pending), BATCH):
                batch = pending[i:i + BATCH]
                mel = torch.stack([mel_of(AUDIO[r["path"]], off, n_mels)
                                   for r in batch]).to("cuda")
                for ts_on in TS_ARMS:
                    if all((name, r["path"], s, ts_on) in done for r in batch):
                        continue
                    res = decode_resilient(model, mel, whisper.DecodingOptions(
                        **DECODE_OPTS, without_timestamps=not ts_on), f"{name}@{s}s")
                    for r, d in zip(batch, res):
                        if (name, r["path"], s, ts_on) in done:
                            continue
                        w.writerow({"model": name, "n_mels": n_mels, "path": r["path"],
                                    "speaker": r["speaker"], "offset_s": s,
                                    "timestamps": "on" if ts_on else "off", "text": d.text,
                                    "avg_logprob": f"{d.avg_logprob:.6f}",
                                    "no_speech_prob": f"{d.no_speech_prob:.6f}"})
                        done.add((name, r["path"], s, ts_on))
                fh.flush()
            print(f"  offset {s:2d}s: {len(pending)} clips x 2 arms in {time.time()-t0:.0f}s")

        del model; gc.collect(); torch.cuda.empty_cache()

PROVENANCE["checkpoints"] = ckpt
PROVENANCE["oom_splits"] = split_events
if split_events:
    print(f"\nnote: batch was split {len(split_events)}x on CUDA OOM -> {split_events[:5]}")
print(f"\nsweep complete: {len(done)} cells in {(time.time()-t_start)/60:.1f} min")

## 6. `delta_m`

Per-utterance WER uses `jiwer.process_words` on a single reference/hypothesis pair, so it can
exceed 1.0 when the model emits more words than the reference - which is exactly what happens in
the runaway cases this metric is built to surface.

Reported three ways, because the distribution is zero-inflated and heavy-tailed and no single
statistic is honest on its own:

- **prevalence** - how many utterances have `delta_m != 0`, split by sign
- **severity** - mean over all clips, and mean over the affected ones only
- **concentration** - what share of the summed positive effect the worst few carry

### Confidence intervals

Three different quantities need three different interval methods here; using one for all of them
would be wrong in at least two places.

- **Mean `delta_m` — BCa bootstrap, 10 000 resamples of size 1000.** Each replicate draws 1000
  utterances with replacement from the 1000 observed. The distribution is strongly right-skewed
  and the naive percentile interval is biased on it, so the endpoints come from BCa-adjusted
  percentiles: a bias correction `z0` plus an acceleration term estimated by jackknife.
- **A speaker-clustered cross-check is reported alongside.** Strictly, utterances from one speaker
  are not independent, and resampling them individually assumes they are. Measured on the local
  `base` run over all 1000 clips the two agree closely — utterance-level
  `[+0.1381, +0.3775]` versus speaker-clustered `[+0.1450, +0.3772]` — because `delta_m` shows
  little within-speaker correlation. Both are printed so the assumption is visible rather than
  buried.
- **Proportions — Wilson score interval.** For `P(delta > 0)` the normal approximation misbehaves
  at small counts and can produce impossible bounds below 0; Wilson stays inside `[0, 1]` and has
  better coverage.
- **Sign test.** An exact two-sided binomial test on positives versus negatives among the
  *affected* utterances. It is distribution-free, so unlike the mean it is untroubled by the heavy
  tail — it asks only whether utterances are more often hurt than helped.

A CI on the median is reported for completeness but will usually be degenerate `[0, 0]`, since
most utterances have `delta_m` exactly 0. That is a fact about the distribution, not a failure.

In [ ]:
import math
from statistics import NormalDist

_N = NormalDist()


def bootstrap_ci(x, groups=None, stat=np.mean, n_boot=10000, alpha=0.05, seed=0):
    """BCa bootstrap CI.

    groups: cluster labels (speaker). When given, whole clusters are resampled instead of
    individual observations, since utterances from one speaker are not independent.
    Returns (point, lo, hi, method).
    """
    x = np.asarray(x, float)
    theta = float(stat(x))
    rng = np.random.default_rng(seed)

    if groups is None:
        # plain resampling: n_boot replicates, each of size len(x), drawn with replacement.
        # Vectorised -- building the (n_boot, n) index matrix once is ~15x faster than
        # concatenating n single-element slices per replicate.
        n = len(x)
        pool = [np.array([i]) for i in range(n)]          # only used by the jackknife below
        idx = rng.integers(0, n, (n_boot, n))
        try:
            boots = np.asarray(stat(x[idx], axis=1), dtype=float)
        except TypeError:                                  # stat has no axis= argument
            boots = np.array([stat(x[row]) for row in idx])
    else:
        g = np.asarray(groups)
        pool = [np.flatnonzero(g == u) for u in np.unique(g)]
        K = len(pool)
        boots = np.empty(n_boot)
        for b in range(n_boot):
            pick = rng.integers(0, K, K)
            boots[b] = stat(np.concatenate([x[pool[i]] for i in pick]))
    K = len(pool)
    boots = np.sort(boots)

    prop = float(np.mean(boots < theta))
    if prop <= 0.0 or prop >= 1.0:          # degenerate, e.g. a median that is always 0
        return theta, float(boots[0]), float(boots[-1]), "percentile(degenerate)"
    z0 = _N.inv_cdf(prop)

    jack = np.empty(K)                       # acceleration via jackknife over clusters
    for i in range(K):
        keep = np.concatenate([pool[j] for j in range(K) if j != i])
        jack[i] = stat(x[keep])
    jbar = jack.mean()
    den = 6.0 * (((jbar - jack) ** 2).sum() ** 1.5)
    a = (((jbar - jack) ** 3).sum() / den) if den != 0 else 0.0

    def adj(p):
        z = _N.inv_cdf(p)
        return _N.cdf(z0 + (z0 + z) / (1 - a * (z0 + z)))

    lo = float(np.quantile(boots, min(max(adj(alpha / 2), 0.0), 1.0)))
    hi = float(np.quantile(boots, min(max(adj(1 - alpha / 2), 0.0), 1.0)))
    return theta, lo, hi, "BCa"


def wilson_ci(k, n, alpha=0.05):
    """Wilson score interval for a proportion: stays inside [0,1], good coverage at small k."""
    if n == 0:
        return float("nan"), 0.0, 1.0
    z = _N.inv_cdf(1 - alpha / 2)
    p, d = k / n, 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, centre - half), min(1.0, centre + half)


def sign_test(pos, neg):
    """Exact two-sided binomial test on the sign split among affected utterances."""
    n = pos + neg
    if n == 0:
        return float("nan")
    k = min(pos, neg)
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)


# self-check against a case with a known answer
_x = np.random.default_rng(1).normal(5, 2, 400)
_t, _lo, _hi, _ = bootstrap_ci(_x, n_boot=2000)
_se = _x.std(ddof=1) / np.sqrt(len(_x))
assert abs(_lo - (_t - 1.96 * _se)) < 0.05 and abs(_hi - (_t + 1.96 * _se)) < 0.05
assert wilson_ci(0, 100)[1] >= 0.0 and sign_test(10, 10) == 1.0
print("CI machinery self-check passed (BCa matches the analytic interval on normal data)")

In [ ]:
from jiwer import process_words

rows = list(csv.DictReader(open(RESULTS_CSV, newline="")))
HYP = {(r["model"], r["path"], int(r["offset_s"]), r["timestamps"]): r["text"] for r in rows}

def wer1(model, path, off, ts):
    return process_words([REF[path]], [normalizer(HYP[(model, path, off, ts)])]).wer

per_utt, summary = [], {}
for name in MODELS:
    d = []
    for r in FILES:
        p = r["path"]
        on  = wer1(name, p, 25, "on")  - wer1(name, p, 5, "on")
        off = wer1(name, p, 25, "off") - wer1(name, p, 5, "off")
        per_utt.append({"model": name, "path": p, "speaker": r["speaker"],
                        "region": r["region"], "sec": f"{r['sec']:.4f}",
                        "n_ref_words": len(REF[p].split()),
                        "wer_on_5":  f"{wer1(name,p,5,'on'):.6f}",
                        "wer_on_25": f"{wer1(name,p,25,'on'):.6f}",
                        "wer_off_5": f"{wer1(name,p,5,'off'):.6f}",
                        "wer_off_25":f"{wer1(name,p,25,'off'):.6f}",
                        "diff_on": f"{on:.6f}", "diff_off": f"{off:.6f}",
                        "delta_m": f"{on-off:.6f}"})
        d.append(on - off)
    d = np.array(d)
    spk = np.array([r["speaker"] for r in FILES])
    pos, neg, zero = (d > 1e-9), (d < -1e-9), (np.abs(d) <= 1e-9)
    top5 = np.sort(d[pos])[::-1][:5].sum() if pos.any() else 0.0

    # primary: 10000 resamples of size len(d), drawn over utterances
    m_hat, m_lo, m_hi, m_meth = bootstrap_ci(d, seed=0)
    md_hat, md_lo, md_hi, _   = bootstrap_ci(d, stat=np.median, seed=0)
    if (~zero).any():
        a_hat, a_lo, a_hi, _ = bootstrap_ci(d[~zero], seed=0)
    else:
        a_hat = a_lo = a_hi = 0.0
    # cross-check: resample the 168 speakers instead, which does not assume utterances
    # within a speaker are independent
    _, mc_lo, mc_hi, _ = bootstrap_ci(d, groups=spk, seed=0)
    p_pos, ppl, pph = wilson_ci(int(pos.sum()), len(d))
    p_aff, pal, pah = wilson_ci(int(pos.sum()), int(pos.sum() + neg.sum()))

    summary[name] = {
        "mean": float(d.mean()), "mean_ci_lo": m_lo, "mean_ci_hi": m_hi, "mean_ci_method": m_meth,
        "mean_ci_lo_clustered": mc_lo, "mean_ci_hi_clustered": mc_hi,
        "median": float(np.median(d)), "median_ci_lo": md_lo, "median_ci_hi": md_hi,
        "std": float(d.std(ddof=1)),
        "n_pos": int(pos.sum()), "n_neg": int(neg.sum()), "n_zero": int(zero.sum()),
        "mean_affected": a_hat, "affected_ci_lo": a_lo, "affected_ci_hi": a_hi,
        "p_pos": p_pos, "p_pos_ci_lo": ppl, "p_pos_ci_hi": pph,
        "p_pos_given_affected": p_aff, "p_aff_ci_lo": pal, "p_aff_ci_hi": pah,
        "sign_test_p": sign_test(int(pos.sum()), int(neg.sum())),
        "top5_share": float(top5 / d[pos].sum()) if pos.any() else float("nan"),
        "max": float(d.max()),
    }

# fallback table; the live value from ckpt[] is preferred when the model was loaded
# fallback table; the live value from ckpt[] is preferred when the model was loaded
PARAMS_TABLE = {"tiny": 39, "base": 74, "small": 244, "medium": 769,
                "large-v1": 1550, "large-v2": 1550, "large-v3": 1550, "large": 1550,
                "large-v3-turbo": 809, "turbo": 809}

def params_of(name):
    """Measured parameter count if we loaded the model, else the table. Never KeyErrors,
    so adding a checkpoint to MODELS cannot crash the summary after the decode work is done."""
    return ckpt.get(name, {}).get("params_M") or PARAMS_TABLE.get(name, 0)

PARAMS = {m: params_of(m) for m in MODELS}

print(f"delta_m over {len(FILES)} utterances")
print(f"95% CIs: BCa bootstrap, {10000} resamples of size {len(FILES)} "
      "(means); Wilson score (proportions)\n")


def ci(lo, hi, sign=True, w=4):
    """Format an interval without nesting quotes inside an f-string (works on Python 3.9+)."""
    fmt = "{:+." + str(w) + "f}" if sign else "{:." + str(w) + "f}"
    return "[" + fmt.format(lo) + ", " + fmt.format(hi) + "]"


print("SEVERITY")
print(f"{'model':>8}{'params':>8}{'mean':>10}{'95% CI':>22}"
      f"{'mean|affected':>15}{'95% CI':>22}{'max':>9}")
print("-" * 94)
for name in MODELS:
    s = summary[name]
    print(f"{name:>8}{PARAMS[name]:>7}M{s['mean']:>+10.4f}"
          f"{ci(s['mean_ci_lo'], s['mean_ci_hi']):>22}"
          f"{s['mean_affected']:>+15.4f}"
          f"{ci(s['affected_ci_lo'], s['affected_ci_hi']):>22}"
          f"{s['max']:>+9.3f}")

print("\nPREVALENCE")
print(f"{'model':>8}{'pos':>6}{'zero':>6}{'neg':>6}{'P(d>0)':>9}{'95% CI':>18}"
      f"{'P(pos|aff)':>12}{'95% CI':>18}{'sign p':>10}")
print("-" * 93)
for name in MODELS:
    s = summary[name]
    print(f"{name:>8}{s['n_pos']:>6}{s['n_zero']:>6}{s['n_neg']:>6}{s['p_pos']:>9.4f}"
          f"{ci(s['p_pos_ci_lo'], s['p_pos_ci_hi'], sign=False):>18}"
          f"{s['p_pos_given_affected']:>12.4f}"
          f"{ci(s['p_aff_ci_lo'], s['p_aff_ci_hi'], sign=False):>18}"
          f"{s['sign_test_p']:>10.5f}")

print("\nspeaker-clustered cross-check on the mean "
      "(resamples 168 speakers, not utterances):")
for name in MODELS:
    s_ = summary[name]
    print(f"  {name:>8}  utterance {ci(s_['mean_ci_lo'], s_['mean_ci_hi'])}"
          f"   speaker {ci(s_['mean_ci_lo_clustered'], s_['mean_ci_hi_clustered'])}")

_sig = [m for m in MODELS if summary[m]["mean_ci_lo"] > 0]
print("\nmean delta_m CI excludes zero for:", _sig if _sig else "no model")
print("median CI is degenerate [0, 0] wherever most utterances are unaffected -- expected")


## 7. Write results, with the licensing guard

Two files. The full record carries hypothesis text and stays on Drive. The per-utterance file is
numbers and paths only - no transcript, no hypothesis - and is asserted to be so before writing,
so it is safe to commit.

In [ ]:
SAFE_FIELDS = ["model", "path", "speaker", "region", "sec", "n_ref_words",
               "wer_on_5", "wer_on_25", "wer_off_5", "wer_off_25",
               "diff_on", "diff_off", "delta_m"]
FORBIDDEN = {"text", "reference", "hypothesis", "ref", "hyp", "transcript"}

assert not (set(SAFE_FIELDS) & FORBIDDEN), "a text-bearing column leaked into SAFE_FIELDS"
assert set(per_utt[0]) == set(SAFE_FIELDS), set(per_utt[0]) ^ set(SAFE_FIELDS)
for row in per_utt:                      # nothing free-text in any value
    for k, v in row.items():
        assert k in ("model", "path", "speaker", "region") or " " not in str(v), (k, v)

with open(SAFE_CSV, "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=SAFE_FIELDS); wr.writeheader(); wr.writerows(per_utt)

PROVENANCE["outputs"] = {
    "full_results": {"path": RESULTS_CSV, "contains_text": True,
                     "sha256": sha256_file(RESULTS_CSV), "git_safe": False},
    "per_utterance": {"path": SAFE_CSV, "contains_text": False,
                      "sha256": sha256_file(SAFE_CSV), "git_safe": True},
}
def _jsonable(v):
    """summary now mixes floats with strings (e.g. the CI method), so coerce per value."""
    if isinstance(v, str):
        return v
    return float(v)

PROVENANCE["summary"] = {k: {kk: _jsonable(vv) for kk, vv in v.items()}
                         for k, v in summary.items()}
with open(PROV_JSON, "w") as f:
    json.dump(PROVENANCE, f, indent=1)

# read back from Drive: proves the write landed rather than sitting in a FUSE buffer
_back = list(csv.DictReader(open(SAFE_CSV, newline="")))
assert len(_back) == len(per_utt) and set(_back[0]) == set(SAFE_FIELDS)
assert abs(float(_back[0]["delta_m"]) - float(per_utt[0]["delta_m"])) < 1e-9

print(f"full record   -> {RESULTS_CSV}   (contains hypotheses; keep out of git)")
print(f"per-utterance -> {SAFE_CSV}   ({len(per_utt)} rows, numbers only; safe to commit)")
print(f"provenance    -> {PROV_JSON}")
print(f"\nchecked: no text-bearing column in the git-safe file")

## 8. Figures

Two views, because prevalence and severity answer different questions and one plot cannot carry
both honestly.

- **Severity** - mean `delta_m` per model with a bootstrap 95% CI. The CI matters here: with a
  handful of huge values driving the mean, an interval built by resampling utterances shows how
  fragile that mean is.
- **Prevalence** - how many of the 300 utterances are affected at all, split by sign. Positive
  means timestamps made the late placement worse for that utterance.

In [ ]:
import matplotlib.pyplot as plt

SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e5e5e2"
C_HURT, C_NONE, C_HELP = "#eb6834", "#c9c8c3", "#2a78d6"   # diverging + neutral midpoint

# reuse the BCa speaker-clustered intervals computed in section 6, so the figure and the
# table can never disagree
boot = {m: (summary[m]["mean"], summary[m]["mean_ci_lo"], summary[m]["mean_ci_hi"])
        for m in MODELS}

# --- severity -------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.2, 4.3), dpi=160)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
ax.grid(True, axis="y", color=GRID, linewidth=1); ax.set_axisbelow(True)
x = np.arange(len(MODELS))
mid = [boot[m][0] for m in MODELS]
lo  = [boot[m][0] - boot[m][1] for m in MODELS]
hi  = [boot[m][2] - boot[m][0] for m in MODELS]
ax.axhline(0, color=INK2, linewidth=1, zorder=2)
ax.errorbar(x, mid, yerr=[lo, hi], fmt="o", color=C_HURT, markersize=8,
            markeredgecolor=SURFACE, markeredgewidth=2, linewidth=2, capsize=5, zorder=3)
ax.plot(x, mid, color=C_HURT, linewidth=2, zorder=2)
for xi, m in zip(x, MODELS):
    ax.annotate(f"{boot[m][0]:+.3f}", (xi, boot[m][2]), textcoords="offset points",
                xytext=(0, 9), ha="center", fontsize=9.5, color=INK)
ax.set_xticks(x); ax.set_xticklabels([f"{m}\n{params_of(m)}M" for m in MODELS])
ax.set_xlim(-0.5, len(MODELS) - 0.5)
ax.set_ylabel("mean  $\\Delta_m$", fontsize=11, color=INK2)
ax.set_title("Timestamp-specific positional penalty, mean per utterance\n"
             "(BCa bootstrap 95% CI, clustered by speaker)",
             fontsize=12, color=INK, pad=12, loc="left")
ax.tick_params(colors=INK2, labelsize=10, length=0)
for s in ("top", "right"): ax.spines[s].set_visible(False)
for s in ("left", "bottom"): ax.spines[s].set_color(GRID); ax.spines[s].set_linewidth(1)
fig.tight_layout(); fig.savefig("delta_severity.png", bbox_inches="tight", facecolor=SURFACE)
fig.savefig(os.path.join(DRIVE_ROOT, "delta_severity.png"), bbox_inches="tight",
            facecolor=SURFACE)

# --- prevalence -----------------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(7.2, 4.3), dpi=160)
fig2.patch.set_facecolor(SURFACE); ax2.set_facecolor(SURFACE)
ax2.grid(True, axis="y", color=GRID, linewidth=1); ax2.set_axisbelow(True)
npos = [summary[m]["n_pos"] for m in MODELS]
nzer = [summary[m]["n_zero"] for m in MODELS]
nneg = [summary[m]["n_neg"] for m in MODELS]
for vals, base, colour, lab in (
        (npos, [0]*len(MODELS), C_HURT, "$\\Delta_m>0$  timestamps hurt"),
        (nzer, npos, C_NONE, "$\\Delta_m=0$  no differential effect"),
        (nneg, [a+b for a, b in zip(npos, nzer)], C_HELP, "$\\Delta_m<0$  timestamps helped")):
    ax2.bar(x, vals, bottom=base, width=0.55, color=colour, label=lab,
            edgecolor=SURFACE, linewidth=2, zorder=3)          # 2px surface gap
for xi, m in zip(x, MODELS):
    ax2.annotate(f"{summary[m]['n_pos']}", (xi, summary[m]["n_pos"] / 2),
                 ha="center", va="center", fontsize=10, color="#ffffff", zorder=4)
ax2.set_xticks(x); ax2.set_xticklabels([f"{m}\n{params_of(m)}M" for m in MODELS])
ax2.set_xlim(-0.5, len(MODELS) - 0.5); ax2.set_ylim(0, len(FILES))
ax2.set_ylabel(f"utterances (of {len(FILES)})", fontsize=11, color=INK2)
ax2.set_title("How many utterances are affected at all", fontsize=12, color=INK,
              pad=12, loc="left")
ax2.tick_params(colors=INK2, labelsize=10, length=0)
for s in ("top", "right"): ax2.spines[s].set_visible(False)
for s in ("left", "bottom"): ax2.spines[s].set_color(GRID); ax2.spines[s].set_linewidth(1)
leg = ax2.legend(frameon=False, fontsize=9, loc="upper right")
for t in leg.get_texts(): t.set_color(INK2)
fig2.tight_layout(); fig2.savefig("delta_prevalence.png", bbox_inches="tight", facecolor=SURFACE)
fig2.savefig(os.path.join(DRIVE_ROOT, "delta_prevalence.png"), bbox_inches="tight",
             facecolor=SURFACE)

print("wrote delta_severity.png and delta_prevalence.png (+ copies on Drive)")
plt.show()

## 9. Verification

In [ ]:
ok = []

N_EXPECTED = SLICE_HI - SLICE_LO
assert len(FILES) == N_EXPECTED and len({r["path"] for r in FILES}) == N_EXPECTED
assert {r["path"] for r in FILES} == {r["path"] for r in DIG["files"][SLICE_LO:SLICE_HI]}
spk = collections.Counter(r["speaker"] for r in FILES)
assert len(spk) == 168
ok.append(f"1. corpus: {N_EXPECTED} clips over {len(spk)} speakers "
          f"({min(spk.values())}-{max(spk.values())} each), every sha256_audio verified")

assert all(o % HOP_LENGTH == 0 for o in OFFSETS)
assert all(o + r["samples"] <= N_SAMPLES for r in FILES for o in OFFSETS)
assert PROVENANCE["mel_gate"]["max_abs_dev"] == 0.0
ok.append("2. offsets: multiples of 160, inside the window, 5 s vs 25 s mel bit-identical")

exp = len(MODELS) * len(FILES) * len(OFFSETS_S) * len(TS_ARMS)
assert len(rows) == exp, f"{len(rows)} rows, expected {exp}"
assert len({(r["model"], r["path"], r["offset_s"], r["timestamps"]) for r in rows}) == exp
ok.append(f"3. grid: {exp} rows, no duplicates, no gaps")

assert all(v["verified"] for v in PROVENANCE["checkpoints"].values())
ok.append("4. checkpoints: all four verified against the SHA-256 in whisper's download URL")

for r in per_utt[:50] + per_utt[-50:]:      # the arithmetic actually holds
    on  = float(r["wer_on_25"])  - float(r["wer_on_5"])
    off = float(r["wer_off_25"]) - float(r["wer_off_5"])
    assert abs(on  - float(r["diff_on"]))  < 1e-6
    assert abs(off - float(r["diff_off"])) < 1e-6
    assert abs((on - off) - float(r["delta_m"])) < 1e-6
ok.append("5. delta_m arithmetic: difference-in-differences recomputed from the stored WERs")

safe = list(csv.DictReader(open(SAFE_CSV, newline="")))
assert len(safe) == len(MODELS) * len(FILES)
assert not (set(safe[0]) & FORBIDDEN)
assert all(" " not in v for r in safe for k, v in r.items() if k not in ("model","path","speaker"))
ok.append("6. licensing: git-safe file has no text column and no free-text values; "
          "hypotheses confined to Drive")

for fn in ("delta_severity.png", "delta_prevalence.png"):
    assert os.path.getsize(fn) > 10000, fn
ok.append("7. figures: both PNGs written")

assert PROVENANCE["corpus"]["reference_digest_sha256"] and PROVENANCE["corpus"]["spec_sha256"]
assert PROVENANCE["code_version"]["whisper_source_sha256_16"]
ok.append("8. provenance: packages, code digests, checkpoints, device, decoding, "
          "corpus + reference digests all recorded")

for name in MODELS:
    s_ = summary[name]
    assert s_["mean_ci_lo"] <= s_["mean"] <= s_["mean_ci_hi"], name
    assert s_["p_pos_ci_lo"] <= s_["p_pos"] <= s_["p_pos_ci_hi"], name
ok.append("9. intervals: every point estimate lies inside its own CI")

# a split means some utterances were decoded at a different batch size from their
# neighbours, which is a reproducibility hazard even if the arithmetic is unaffected
if split_events:
    print(f"WARN  batch was split {len(split_events)}x on CUDA OOM: {split_events[:5]}")
    print("      results are still valid but not uniformly batched; lower the entry for "
          "BATCH and re-run that model for a clean record")
else:
    _b = {ckpt[m]["batch"] for m in ckpt}
    assert _b == {BATCH} and len(FILES) % BATCH == 0
    ok.append(f"10. batching: every model at batch {BATCH}, "
              f"{len(FILES)//BATCH} full batches, no remainder, no OOM splits")

for line in ok:
    print("PASS  " + line)
print(f"\nprovenance written to {PROV_JSON}")

## 10. Doing statistics later, without a GPU

Everything needed for further analysis is on Drive, so the runtime can be disconnected as soon as
section 7 has run. Three files persist under `MyDrive/NAACL/`:

| file | contents | needed for |
|---|---|---|
| `delta_per_utterance.csv` | one row per (model, utterance) for all 1000 clips: `delta_m`, the four component WERs, both inner differences, plus `speaker`, `region`, `sec`, `n_ref_words` | **any statistic you decide on later** |
| `delta_results_full.csv` | the raw hypotheses, appended batch by batch during the sweep | recomputing WER under a different normalizer or scoring rule |
| `delta_provenance.json` | versions, checkpoint digests, decoding options, corpus + reference digests, and the summary with CIs | reporting and audit |

`delta_per_utterance.csv` deliberately carries more than `delta_m` alone. Because it keeps
`n_ref_words`, per-file WERs can be converted back to error counts and re-pooled into a
corpus-level figure — which a file of WER ratios alone could not support. `speaker` supports
clustered resampling, and `region` and `sec` support covariate analysis (does the penalty grow
with utterance duration? does it differ by dialect region?).

The cell below is a **standalone entry point**: it needs no GPU, no TIMIT, and none of the earlier
cells. Run it in a fresh CPU runtime to reload and start analysing.

### One caveat about Drive

Colab's Drive mount is a FUSE layer, and a write is not guaranteed to have reached Google Drive
the instant `flush()` returns. Section 7 reads `delta_per_utterance.csv` back and compares it to
the in-memory rows, which proves the bytes landed. If you kill a runtime abruptly mid-sweep, the
last batch or two of `delta_results_full.csv` may not have synced — harmless, because the sweep
skips whatever it finds and simply redoes the rest.

In [ ]:
# --- standalone: run this alone in a fresh CPU runtime -----------------------
# from google.colab import drive; drive.mount("/content/drive")
import csv, json

SAFE = "/content/drive/MyDrive/NAACL/delta_per_utterance.csv"
PROV = "/content/drive/MyDrive/NAACL/delta_provenance.json"

rows_ = list(csv.DictReader(open(SAFE, newline="")))
NUM = ("sec", "n_ref_words", "wer_on_5", "wer_on_25", "wer_off_5", "wer_off_25",
       "diff_on", "diff_off", "delta_m")
for r_ in rows_:
    for k_ in NUM:
        r_[k_] = float(r_[k_])

models_ = sorted({r_["model"] for r_ in rows_})
print(f"{len(rows_)} rows | models {models_} | "
      f"{len({r_['path'] for r_ in rows_})} utterances | "
      f"{len({r_['speaker'] for r_ in rows_})} speakers")
print("columns:", list(rows_[0]))

prov_ = json.load(open(PROV))
print(f"\nrun: whisper {prov_['packages']['openai-whisper']} on {prov_['device']['name']}, "
      f"{prov_['device']['precision']}")
print(f"corpus digest {prov_['corpus']['corpus_digest_sha256'][:16]}... | "
      f"reference digest {prov_['corpus']['reference_digest_sha256'][:16]}...")

# example: corpus-level WER re-pooled from per-file WERs, which needs n_ref_words
for m_ in models_:
    sel_ = [r_ for r_ in rows_ if r_["model"] == m_]
    tot_ = sum(r_["n_ref_words"] for r_ in sel_)
    pooled_on = sum(r_["wer_on_25"] * r_["n_ref_words"] for r_ in sel_) / tot_
    pooled_off = sum(r_["wer_off_25"] * r_["n_ref_words"] for r_ in sel_) / tot_
    mean_d = sum(r_["delta_m"] for r_ in sel_) / len(sel_)
    print(f"  {m_:>9}: pooled WER@25s on {pooled_on:.4f} / off {pooled_off:.4f} | "
          f"mean delta_m {mean_d:+.4f}")

# with pandas, if you prefer:
#   import pandas as pd; df = pd.read_csv(SAFE)
#   df.groupby("model")["delta_m"].describe()